In [1]:
# Hi-SAM
import os

REPO_DIR = "/content/Hi-SAM"

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/ymy-k/Hi-SAM.git {REPO_DIR}

# Hi-SAM 공식 requirements 전체 설치
%pip -q install -r /content/Hi-SAM/requirements.txt

os.makedirs(f"{REPO_DIR}/pretrained_checkpoint", exist_ok=True)

!wget -q -c -O /content/Hi-SAM/pretrained_checkpoint/sam_vit_l_0b3195.pth \
    https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth

!wget -q -c -O /content/Hi-SAM/pretrained_checkpoint/hi_sam_l.pth \
    https://huggingface.co/GoGiants1/Hi-SAM/resolve/main/hi_sam_l.pth

  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [2]:
# Qwen2.5-VL
!pip -q uninstall -y transformers
!pip -q install git+https://github.com/huggingface/transformers accelerate qwen-vl-utils pillow

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [35]:
import os
import sys
import re
import json
import math
import random
from pathlib import Path
from types import SimpleNamespace

import cv2
import torch
import numpy as np
from PIL import Image, ImageDraw
from shapely.geometry import Polygon
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

try:
    import pyclipper
except ModuleNotFoundError:
    !pip -q install pyclipper
    import pyclipper


# =========================================================
# 0. CONFIG
# =========================================================
REPO_DIR = "/content/Hi-SAM"
IMAGE_PATH = "/content/drive/MyDrive/Colab Notebooks/course_1.jpg"
SAVE_DIR = "/content/drive/MyDrive/Colab Notebooks"

# Hi-SAM
MODEL_TYPE = "vit_l"
CHECKPOINT = f"{REPO_DIR}/pretrained_checkpoint/hi_sam_l.pth"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TOTAL_POINTS = 1800
BATCH_POINTS = 100
SCORE_THRESH = 0.4
NMS_THRESH = 0.5
MIN_POLYGON_AREA = 32

# Crop / Tile
CROP_MARGIN = 20
UPSCALE = 2
TILE_BATCH_SIZE = 9
TILE_COLUMNS = 3
TILE_CELL_PADDING = 16
TILE_OUTER_PADDING = 24
TILE_BG_COLOR = (255, 255, 255)
TILE_DEBUG_SAVE = True

# Final output
DROP_EMPTY_TEXT = False   # True면 text="" 항목 제거
SORT_BY_COORDS = True     # True면 y, x 순 정렬
DROP_EMPTY_TEXT = True    # True면 최종 출력에서 제외

# Qwen2.5-VL
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
MAX_NEW_TOKENS = 512

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

os.makedirs(SAVE_DIR, exist_ok=True)

POLYGON_JSON_PATH = os.path.join(SAVE_DIR, "course_1_word_polygons.json")
FINAL_JSON_PATH = os.path.join(SAVE_DIR, "course_1_text_xy.json")
TILE_DIR = os.path.join(SAVE_DIR, "debug_tiles")

In [28]:
# =========================================================
# 1. Hi-SAM UTILS
# =========================================================
def unclip(p, unclip_ratio=3.5):
    poly = Polygon(p)
    distance = poly.area * unclip_ratio / max(poly.length, 1e-6)
    offset = pyclipper.PyclipperOffset()
    offset.AddPath(p.astype(np.int32), pyclipper.JT_ROUND, pyclipper.ET_CLOSEDPOLYGON)
    expanded = np.array(offset.Execute(distance))
    return expanded


def mask_to_polygon_list(mask, img_h, img_w, min_area=32):
    mask = mask.astype(np.uint8)
    polygons = []

    contours, _ = cv2.findContours(mask, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)

    for cont in contours:
        epsilon = 0.002 * cv2.arcLength(cont, True)
        approx = cv2.approxPolyDP(cont, epsilon, True)
        points = approx.reshape((-1, 2))

        if points.shape[0] < 4:
            continue

        try:
            pts = unclip(points)
            if len(pts) != 1:
                continue
            pts = pts[0].astype(np.int32)
        except Exception:
            pts = points.astype(np.int32)

        if len(pts) < 4:
            continue

        area = Polygon(pts).area
        if area < min_area:
            continue

        pts[:, 0] = np.clip(pts[:, 0], 0, img_w - 1)
        pts[:, 1] = np.clip(pts[:, 1], 0, img_h - 1)

        polygons.append(pts)

    return polygons


def run_hisam_to_json(
    image_path,
    json_path,
    repo_dir,
    checkpoint,
    model_type="vit_l",
    device="cuda",
    total_points=1800,
    batch_points=100,
    score_thresh=0.5,
    nms_thresh=0.5,
    min_polygon_area=32,
):
    assert os.path.exists(image_path), f"Image not found: {image_path}"
    assert os.path.exists(checkpoint), f"Checkpoint not found: {checkpoint}"

    os.chdir(repo_dir)
    if repo_dir not in sys.path:
        sys.path.append(repo_dir)

    from hi_sam.modeling.build import model_registry
    from hi_sam.modeling.auto_mask_generator import AutoMaskGenerator

    args = SimpleNamespace(
        checkpoint=checkpoint,
        model_type=model_type,
        device=device,
        hier_det=True,
        input_size=[1024, 1024],
        attn_layers=1,
        prompt_len=12,
        layout_thresh=0.5,
    )

    hisam = model_registry[model_type](args)
    hisam.eval().to(device)

    efficient_hisam = model_type in ["vit_s", "vit_t"]
    amg = AutoMaskGenerator(hisam, efficient_hisam=efficient_hisam)

    image_bgr = cv2.imread(image_path)
    assert image_bgr is not None, f"Failed to read image: {image_path}"
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    img_h, img_w = image_rgb.shape[:2]

    amg.set_image(image_rgb)

    with torch.inference_mode():
        masks, scores, affinity = amg.predict(
            from_low_res=False,
            fg_points_num=total_points,
            batch_points_num=batch_points,
            score_thresh=score_thresh,
            nms_thresh=nms_thresh,
        )

    if masks is None:
        raise RuntimeError("No word masks were predicted.")

    word_masks = (masks[:, 0, :, :]).astype(np.uint8)

    all_word_polygons = []
    for wm in word_masks:
        polygons = mask_to_polygon_list(
            wm, img_h, img_w, min_area=min_polygon_area
        )
        for poly in polygons:
            all_word_polygons.append({
                "id": len(all_word_polygons),
                "vertices": poly.tolist(),
            })

    payload = {
        "image_path": image_path,
        "image_width": img_w,
        "image_height": img_h,
        "num_words": len(all_word_polygons),
        "words": all_word_polygons,
    }

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    print(f"[Hi-SAM] saved json: {json_path}")
    print(f"[Hi-SAM] num_words: {len(all_word_polygons)}")

    del hisam
    del amg
    import gc
    gc.collect()
    torch.cuda.empty_cache()

    return payload

In [29]:
# =========================================================
# 2. CROP IN MEMORY + CENTER COORDS
# =========================================================
def polygon_to_bbox(vertices, img_w, img_h, margin=8):
    xs = [p[0] for p in vertices]
    ys = [p[1] for p in vertices]

    x1 = max(0, min(xs) - margin)
    y1 = max(0, min(ys) - margin)
    x2 = min(img_w, max(xs) + margin)
    y2 = min(img_h, max(ys) + margin)

    return [int(x1), int(y1), int(x2), int(y2)]


def polygon_center_xy(vertices, img_w, img_h):
    try:
        poly = Polygon(vertices)
        if poly.is_valid and poly.area > 0:
            c = poly.centroid
            cx = int(round(c.x))
            cy = int(round(c.y))
            cx = max(0, min(cx, img_w - 1))
            cy = max(0, min(cy, img_h - 1))
            return cx, cy
    except Exception:
        pass

    xs = [p[0] for p in vertices]
    ys = [p[1] for p in vertices]
    cx = int(round(sum(xs) / len(xs)))
    cy = int(round(sum(ys) / len(ys)))
    cx = max(0, min(cx, img_w - 1))
    cy = max(0, min(cy, img_h - 1))
    return cx, cy


def build_crop_records_from_json(json_data, margin=8, upscale=2):
    image_path = json_data["image_path"]
    img = Image.open(image_path).convert("RGB")
    img_w, img_h = img.size

    crop_records = []

    for item in json_data["words"]:
        wid = item["id"]
        vertices = item["vertices"]

        x1, y1, x2, y2 = polygon_to_bbox(vertices, img_w, img_h, margin=margin)
        center_x, center_y = polygon_center_xy(vertices, img_w, img_h)

        crop = img.crop((x1, y1, x2, y2))

        if upscale != 1:
            crop = crop.resize(
                (crop.width * upscale, crop.height * upscale),
                Image.Resampling.LANCZOS
            )

        crop_records.append({
            "id": wid,
            "bbox": [x1, y1, x2, y2],
            "center_x": center_x,
            "center_y": center_y,
            "vertices": vertices,
            "crop_image": crop,
            "crop_width": crop.width,
            "crop_height": crop.height,
        })

    return crop_records


In [30]:
# =========================================================
# 3. TILE IMAGE BUILDING
# =========================================================
def chunk_list(lst, chunk_size):
    for i in range(0, len(lst), chunk_size):
        yield lst[i:i + chunk_size]


def make_tile_image(
    crop_records_chunk,
    columns=3,
    cell_padding=16,
    outer_padding=24,
    bg_color=(255, 255, 255)
):
    assert len(crop_records_chunk) > 0

    rows = math.ceil(len(crop_records_chunk) / columns)

    max_w = max(r["crop_width"] for r in crop_records_chunk)
    max_h = max(r["crop_height"] for r in crop_records_chunk)

    cell_w = max_w + cell_padding * 2
    cell_h = max_h + cell_padding * 2

    tile_w = outer_padding * 2 + columns * cell_w
    tile_h = outer_padding * 2 + rows * cell_h

    tile = Image.new("RGB", (tile_w, tile_h), color=bg_color)
    draw = ImageDraw.Draw(tile)

    mapping = []

    for idx, rec in enumerate(crop_records_chunk):
        row = idx // columns
        col = idx % columns

        cell_x1 = outer_padding + col * cell_w
        cell_y1 = outer_padding + row * cell_h
        cell_x2 = cell_x1 + cell_w
        cell_y2 = cell_y1 + cell_h

        crop = rec["crop_image"]

        paste_x = cell_x1 + (cell_w - crop.width) // 2
        paste_y = cell_y1 + (cell_h - crop.height) // 2

        tile.paste(crop, (paste_x, paste_y))
        draw.rectangle(
            [cell_x1, cell_y1, cell_x2 - 1, cell_y2 - 1],
            outline=(220, 220, 220),
            width=1
        )

        mapping.append({
            "cell_index": idx,
            "row": row,
            "col": col,
            "id": rec["id"],
            "bbox": rec["bbox"],
            "center_x": rec["center_x"],
            "center_y": rec["center_y"],
            "tile_cell_box": [cell_x1, cell_y1, cell_x2, cell_y2],
        })

    tile_meta = {
        "num_cells": len(crop_records_chunk),
        "columns": columns,
        "rows": rows,
        "cell_width": cell_w,
        "cell_height": cell_h,
        "mapping": mapping,
    }
    return tile, tile_meta


def build_all_tiles(
    crop_records,
    batch_size=9,
    columns=3,
    cell_padding=16,
    outer_padding=24,
    bg_color=(255, 255, 255)
):
    all_tiles = []

    for chunk_idx, chunk in enumerate(chunk_list(crop_records, batch_size)):
        tile_img, tile_meta = make_tile_image(
            chunk,
            columns=columns,
            cell_padding=cell_padding,
            outer_padding=outer_padding,
            bg_color=bg_color,
        )
        all_tiles.append({
            "tile_index": chunk_idx,
            "tile_image": tile_img,
            "tile_meta": tile_meta,
        })

    return all_tiles

In [31]:
# =========================================================
# 4. QWEN OCR
# =========================================================
def load_qwen_model(model_id="Qwen/Qwen2.5-VL-7B-Instruct"):
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="sdpa",
    )
    processor = AutoProcessor.from_pretrained(model_id)
    return model, processor


def build_qwen_prompt_for_tile(num_cells):
    return f"""
이 이미지는 여러 개의 텍스트 crop을 격자(grid) 형태로 배치한 타일 이미지다.
셀의 순서는 반드시 row-major order(왼쪽에서 오른쪽, 위에서 아래)로 해석해라.
즉 첫 번째 행을 왼쪽부터 읽고, 다음 행으로 넘어간다.

총 셀 개수는 {num_cells}개다.
각 셀마다 보이는 텍스트를 읽어서 JSON 배열만 출력해라.
반드시 아래 형식을 지켜라.

[
  {{"cell_index": 0, "text": "..."}},
  {{"cell_index": 1, "text": "..."}}
]

규칙:
- JSON 배열만 출력한다.
- 설명, 해설, 마크다운 코드블록을 절대 넣지 마라.
- 텍스트가 확실하지 않으면 가장 가능성 높은 값을 쓰되, 빈 셀이면 빈 문자열 ""을 넣어라.
- 한국어, 영어, 숫자, 기호를 모두 그대로 보존해라.
- 각 셀의 텍스트는 해당 셀 안에 있는 내용만 적어라.
- 반드시 0부터 {num_cells - 1}까지의 모든 cell_index를 하나씩 포함해라.
""".strip()


def run_qwen_on_single_tile(tile_image, tile_meta, model, processor, max_new_tokens=512):
    prompt = build_qwen_prompt_for_tile(tile_meta["num_cells"])

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": tile_image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    inputs = inputs.to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    return output_text


In [32]:
# =========================================================
# 5. OUTPUT PARSING
# =========================================================
def extract_json_array(text):
    text = text.strip()
    text = re.sub(r"^```json\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    start = text.find("[")
    end = text.rfind("]")
    if start != -1 and end != -1 and end > start:
        return text[start:end + 1]

    return text


def parse_qwen_tile_output(output_text, expected_num_cells):
    cleaned = extract_json_array(output_text)

    try:
        arr = json.loads(cleaned)
    except Exception:
        return {
            "ok": False,
            "parsed": [{"cell_index": i, "text": ""} for i in range(expected_num_cells)]
        }

    parsed = []
    seen = set()

    if isinstance(arr, list):
        for item in arr:
            if not isinstance(item, dict):
                continue
            if "cell_index" not in item:
                continue

            try:
                idx = int(item["cell_index"])
            except Exception:
                continue

            if idx < 0 or idx >= expected_num_cells:
                continue

            txt = str(item.get("text", ""))
            parsed.append({"cell_index": idx, "text": txt})
            seen.add(idx)

    for i in range(expected_num_cells):
        if i not in seen:
            parsed.append({"cell_index": i, "text": ""})

    parsed = sorted(parsed, key=lambda x: x["cell_index"])

    return {
        "ok": True,
        "parsed": parsed
    }

In [33]:
# =========================================================
# 6. MAP OCR RESULT -> TEXT + CENTER COORDS
# =========================================================
def merge_tile_result_to_text_xy(tile_meta, parsed_result):
    cell_to_meta = {m["cell_index"]: m for m in tile_meta["mapping"]}
    merged = []

    for item in parsed_result["parsed"]:
        cell_index = item["cell_index"]
        text = item["text"]

        meta = cell_to_meta.get(cell_index)
        if meta is None:
            continue

        merged.append({
            "text": text,
            "x": int(meta["center_x"]),
            "y": int(meta["center_y"]),
        })

    return merged


In [36]:
# =========================================================
# 7. FULL PIPELINE
# =========================================================
def run_full_pipeline():
    # -----------------------------------------------------
    # Step 1. Hi-SAM -> polygon json only
    # -----------------------------------------------------
    hisam_json = run_hisam_to_json(
        image_path=IMAGE_PATH,
        json_path=POLYGON_JSON_PATH,
        repo_dir=REPO_DIR,
        checkpoint=CHECKPOINT,
        model_type=MODEL_TYPE,
        device=DEVICE,
        total_points=TOTAL_POINTS,
        batch_points=BATCH_POINTS,
        score_thresh=SCORE_THRESH,
        nms_thresh=NMS_THRESH,
        min_polygon_area=MIN_POLYGON_AREA,
    )

    # -----------------------------------------------------
    # Step 2. polygon json + original image -> memory crops
    # -----------------------------------------------------
    crop_records = build_crop_records_from_json(
        hisam_json,
        margin=CROP_MARGIN,
        upscale=UPSCALE,
    )
    print(f"[CROP] num crops in memory: {len(crop_records)}")

    # -----------------------------------------------------
    # Step 3. build tile images
    # -----------------------------------------------------
    all_tiles = build_all_tiles(
        crop_records,
        batch_size=TILE_BATCH_SIZE,
        columns=TILE_COLUMNS,
        cell_padding=TILE_CELL_PADDING,
        outer_padding=TILE_OUTER_PADDING,
        bg_color=TILE_BG_COLOR,
    )
    print(f"[TILE] num tiles: {len(all_tiles)}")

    if TILE_DEBUG_SAVE:
        os.makedirs(TILE_DIR, exist_ok=True)
        for tile_item in all_tiles:
            tile_path = os.path.join(TILE_DIR, f"tile_{tile_item['tile_index']:03d}.png")
            tile_item["tile_image"].save(tile_path)
        print(f"[TILE] debug tiles saved to: {TILE_DIR}")

    # -----------------------------------------------------
    # Step 4. load Qwen model
    # -----------------------------------------------------
    model, processor = load_qwen_model(MODEL_ID)

    # -----------------------------------------------------
    # Step 5. OCR each tile
    # -----------------------------------------------------
    final_results = []

    for tile_item in all_tiles:
        tile_index = tile_item["tile_index"]
        tile_image = tile_item["tile_image"]
        tile_meta = tile_item["tile_meta"]

        print(f"[QWEN] tile {tile_index} / cells={tile_meta['num_cells']}")

        raw_output = run_qwen_on_single_tile(
            tile_image=tile_image,
            tile_meta=tile_meta,
            model=model,
            processor=processor,
            max_new_tokens=MAX_NEW_TOKENS,
        )

        parsed = parse_qwen_tile_output(
            raw_output,
            expected_num_cells=tile_meta["num_cells"]
        )

        merged = merge_tile_result_to_text_xy(tile_meta, parsed)
        final_results.extend(merged)

    # -----------------------------------------------------
    # Step 6. free Qwen GPU memory
    # -----------------------------------------------------
    del model
    del processor
    import gc
    gc.collect()
    torch.cuda.empty_cache()

    # -----------------------------------------------------
    # Step 7. optional filtering / sorting
    # -----------------------------------------------------
    if DROP_EMPTY_TEXT:
        final_results = [
            r for r in final_results
            if isinstance(r["text"], str) and r["text"].strip() != ""
        ]

    if SORT_BY_COORDS:
        final_results = sorted(final_results, key=lambda r: (r["y"], r["x"]))

    # -----------------------------------------------------
    # Step 8. save final output only
    # -----------------------------------------------------
    final_payload = {
        "image_path": IMAGE_PATH,
        "num_texts": len(final_results),
        "results": final_results,
        "config": {
            "crop_margin": CROP_MARGIN,
            "upscale": UPSCALE,
            "tile_batch_size": TILE_BATCH_SIZE,
            "tile_columns": TILE_COLUMNS,
            "tile_cell_padding": TILE_CELL_PADDING,
            "tile_outer_padding": TILE_OUTER_PADDING,
            "model_id": MODEL_ID,
            "drop_empty_text": DROP_EMPTY_TEXT,
            "sort_by_coords": SORT_BY_COORDS,
        }
    }

    with open(FINAL_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(final_payload, f, ensure_ascii=False, indent=2)

    print(f"[DONE] final output saved to: {FINAL_JSON_PATH}")
    return final_payload


# =========================================================
# 8. RUN
# =========================================================
results = run_full_pipeline()

print("\n[Preview]")
for item in results["results"][:10]:
    print(item)

Freeze image encoder.
<All keys matched successfully>


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


[Hi-SAM] saved json: /content/drive/MyDrive/Colab Notebooks/course_1_word_polygons.json
[Hi-SAM] num_words: 42
[CROP] num crops in memory: 42
[TILE] num tiles: 5
[TILE] debug tiles saved to: /content/drive/MyDrive/Colab Notebooks/debug_tiles


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

[QWEN] tile 0 / cells=9
[QWEN] tile 1 / cells=9
[QWEN] tile 2 / cells=9
[QWEN] tile 3 / cells=9
[QWEN] tile 4 / cells=6
[DONE] final output saved to: /content/drive/MyDrive/Colab Notebooks/course_1_text_xy.json

[Preview]
{'text': '중구보건소', 'x': 440, 'y': 20}
{'text': '중부경찰서', 'x': 350, 'y': 43}
{'text': '1.5km', 'x': 433, 'y': 64}
{'text': '학성지하차도', 'x': 434, 'y': 83}
{'text': '울산종합운동장', 'x': 287, 'y': 85}
{'text': '전방', 'x': 286, 'y': 113}
{'text': '메가마트', 'x': 476, 'y': 124}
{'text': '울산종합운동장', 'x': 878, 'y': 131}
{'text': 'START FINISH', 'x': 287, 'y': 139}
{'text': '학성공원', 'x': 463, 'y': 157}
